# Sea Ice Diagnostics and LENS comparison

This notebook contains:
- Statistics of Ice Area, Ice Volume, and Snow Volume

In [ ]:
import os

import xarray as xr
import statistics
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import nc_time_axis

import cupid_utils as cu

In [ ]:
CESM_output_dir = (
    []
)  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"
ts_dir = None  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"

case_names = []
case_nicknames = []
start_dates = []
end_dates = []
case_align_years = []

obs_data_dir = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_obs_data"
path_model = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_model_data/ice/"
grid_file = ""  # "/glade/campaign/cesm/community/omwg/grids/tx2_3v2_grid.nc"
climo_nyears = 35

serial = False  # use dask LocalCluster

lc_kwargs = {}

In [ ]:
# Want some base case parameter defaults to equal control case values
if ts_dir is None:
    ts_dir = CESM_output_dir

In [ ]:
# When running interactively, cupid_run should be set to 0 for
# a DASK cluster

cupid_run = 1

if cupid_run == 1:

    from dask.distributed import Client, LocalCluster

    # Spin up cluster (if running in parallel)
    client = None
    if not serial:
        cluster = LocalCluster(**lc_kwargs)
        client = Client(cluster)

else:

    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    cluster = PBSCluster(
        cores=16,
        processes=16,
        memory="100GB",
        account="P93300065",
        queue="casper",
        walltime="02:00:00",
    )

    client = Client(cluster)

    cluster.scale(1)

    print(cluster)

client

In [ ]:
ds_grid = xr.open_dataset(grid_file)

if "TLAT" in ds_grid:
    TLAT = ds_grid["TLAT"]
    TLON = ds_grid["TLONG"]
    tarea = ds_grid["TAREA"] * 1.0e-4
    angle = ds_grid["ANGLE"]
else:
    TLAT = ds_grid["tlat"] * 180.0 / np.pi
    TLON = ds_grid["tlon"] * 180.0 / np.pi
    htn = ds_grid["htn"] * 1.0e-2
    hte = ds_grid["hte"] * 1.0e-2
    tarea = htn * hte
    angle = ds_grid["angle"]

yml_dir = os.path.join(cu.__path__[0], "..", "input_files", "ice")
with open(os.path.join(yml_dir, "cice_masks.yml"), "r") as file:
    cice_masks = yaml.safe_load(file)

path_lens1 = os.path.join(path_model, "cesm_lens1")
path_lens2 = os.path.join(path_model, "cesm_lens2")

path_cdr = os.path.join(
    obs_data_dir, "ice", "analysis_datasets", "hemispheric_data", "CDR_area_timeseries/"
)

In [ ]:
## set up dictionary for all cases listed

cmap = plt.get_cmap("Set1")
colors = cmap.colors[: len(case_names)]

case_info = {}
for case_name, case_nickname, start_date, end_date, case_align_year, tsdir in zip(
    case_names, case_nicknames, start_dates, end_dates, case_align_years, ts_dir
):

    ds = xr.open_mfdataset(
        os.path.join(
            tsdir, case_name, "ice", "proc", "tseries", f"{case_name}.cice.h.*.nc"
        ),
        data_vars="minimal",
        compat="override",
        coords="minimal",
    ).sel(time=slice(start_date, end_date))

    ds_ann = ds.resample(time="YS").mean(dim="time")
    climo_n_years = min(
        climo_nyears, len(ds_ann.time)
    )  ## had to name this something different than global climo_nyears set in params

    first_year = int(start_date.split("-")[0])
    end_year = int(end_date.split("-")[0])

    case_info[case_name] = {
        "case_nickname": case_nickname,
        "start_date": start_date,
        "end_date": end_date,
        "first_year": first_year,
        "end_year": end_year,
        "case_align_year": case_align_year,
        "ds": ds,
        "ds_ann": ds_ann,
        "climo_nyears": climo_n_years,
        "color": tuple(colors[case_names.index(case_name)]),
    }

In [ ]:
### Read in the CESM LENS historical data

ds_cesm1_aicetot_nh = xr.open_dataset(path_lens1 + "/LE_aicetot_nh_1920-2100.nc")
ds_cesm1_hitot_nh = xr.open_dataset(path_lens1 + "/LE_hitot_nh_1920-2100.nc")
ds_cesm1_hstot_nh = xr.open_dataset(path_lens1 + "/LE_hstot_nh_1920-2100.nc")

ds_cesm1_aicetot_sh = xr.open_dataset(path_lens1 + "/LE_aicetot_sh_1920-2100.nc")
ds_cesm1_hitot_sh = xr.open_dataset(path_lens1 + "/LE_hitot_sh_1920-2100.nc")
ds_cesm1_hstot_sh = xr.open_dataset(path_lens1 + "/LE_hstot_sh_1920-2100.nc")

cesm1_aicetot_nh_ann = ds_cesm1_aicetot_nh["aice_monthly"].mean(dim="nmonth")
cesm1_hitot_nh_ann = ds_cesm1_hitot_nh["hi_monthly"].mean(dim="nmonth")
cesm1_hstot_nh_ann = ds_cesm1_hstot_nh["hs_monthly"].mean(dim="nmonth")

cesm1_aicetot_sh_ann = ds_cesm1_aicetot_sh["aice_monthly"].mean(dim="nmonth")
cesm1_hitot_sh_ann = ds_cesm1_hitot_sh["hi_monthly"].mean(dim="nmonth")
cesm1_hstot_sh_ann = ds_cesm1_hstot_sh["hs_monthly"].mean(dim="nmonth")

if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm1_years = np.linspace(1920, 2100, 181)
else:
    cesm1_years = np.linspace(1, 181, 181)

if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm1_climo_start = 60
    cesm1_climo_end = 95
else:
    cesm1_climo_start = 10
    cesm1_climo_end = 45

cesm1_aicetot_nh_month = ds_cesm1_aicetot_nh["aice_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hitot_nh_month = ds_cesm1_hitot_nh["hi_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hstot_nh_month = ds_cesm1_hstot_nh["hs_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_aicetot_sh_month = ds_cesm1_aicetot_sh["aice_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hitot_sh_month = ds_cesm1_hitot_sh["hi_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")
cesm1_hstot_sh_month = ds_cesm1_hstot_sh["hs_monthly"][
    :, cesm1_climo_start:cesm1_climo_end, :
].mean(dim="nyr")

ds_cesm2_aicetot_nh = xr.open_dataset(path_lens2 + "/LE2_aicetot_nh_1870-2100.nc")
ds_cesm2_hitot_nh = xr.open_dataset(path_lens2 + "/LE2_hitot_nh_1870-2100.nc")
ds_cesm2_hstot_nh = xr.open_dataset(path_lens2 + "/LE2_hstot_nh_1870-2100.nc")

ds_cesm2_aicetot_sh = xr.open_dataset(path_lens2 + "/LE2_aicetot_sh_1870-2100.nc")
ds_cesm2_hitot_sh = xr.open_dataset(path_lens2 + "/LE2_hitot_sh_1870-2100.nc")
ds_cesm2_hstot_sh = xr.open_dataset(path_lens2 + "/LE2_hstot_sh_1870-2100.nc")

cesm2_aicetot_nh_ann = ds_cesm2_aicetot_nh["aice_monthly"].mean(dim="nmonth")
cesm2_hitot_nh_ann = ds_cesm2_hitot_nh["hi_monthly"].mean(dim="nmonth")
cesm2_hstot_nh_ann = ds_cesm2_hstot_nh["hs_monthly"].mean(dim="nmonth")

cesm2_aicetot_sh_ann = ds_cesm2_aicetot_sh["aice_monthly"].mean(dim="nmonth")
cesm2_hitot_sh_ann = ds_cesm2_hitot_sh["hi_monthly"].mean(dim="nmonth")
cesm2_hstot_sh_ann = ds_cesm2_hstot_sh["hs_monthly"].mean(dim="nmonth")

if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm2_years = np.linspace(1870, 2100, 231)
else:
    cesm2_years = np.linspace(1, 231, 231)

if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm2_climo_start = 110
    cesm2_climo_end = 145
else:
    cesm2_climo_start = 61
    cesm2_climo_end = 96

cesm2_aicetot_nh_month = ds_cesm2_aicetot_nh["aice_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hitot_nh_month = ds_cesm2_hitot_nh["hi_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hstot_nh_month = ds_cesm2_hstot_nh["hs_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_aicetot_sh_month = ds_cesm2_aicetot_sh["aice_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hitot_sh_month = ds_cesm2_hitot_sh["hi_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")
cesm2_hstot_sh_month = ds_cesm2_hstot_sh["hs_monthly"][
    :, cesm2_climo_start:cesm2_climo_end, :
].mean(dim="nyr")

In [ ]:
cdr_nh = xr.open_dataset(path_cdr + "CDR_sic_nh_monthly.nc")
cdr_sh = xr.open_dataset(path_cdr + "CDR_sic_sh_monthly.nc")
cdr_lab = xr.open_dataset(path_cdr + "CDR_sic_lab_monthly.nc")

cdr_nh_mar = cdr_nh["sic_monthly"].sel(time=(cdr_nh.time.dt.month == 3))
cdr_sh_feb = cdr_sh["sic_monthly"].sel(time=(cdr_sh.time.dt.month == 2))
cdr_nh_sep = cdr_nh["sic_monthly"].sel(time=(cdr_nh.time.dt.month == 9))
cdr_sh_sep = cdr_sh["sic_monthly"].sel(time=(cdr_sh.time.dt.month == 9))
cdr_lab_mar = cdr_lab["sic_monthly_lab"].sel(time=(cdr_lab.time.dt.month == 3))

cdr_nh_clim = (
    cdr_nh["sic_monthly"]
    .isel(time=slice(-420, None))
    .groupby("time.month")
    .mean(dim="time", skipna=True)
)

cdr_sh_clim = (
    cdr_sh["sic_monthly"]
    .isel(time=slice(-420, None))
    .groupby("time.month")
    .mean(dim="time", skipna=True)
)

In [ ]:
cesm1_aicetot_nh_mar = ds_cesm1_aicetot_nh["aice_monthly"].isel(nmonth=3)
cesm1_hitot_nh_mar = ds_cesm1_hitot_nh["hi_monthly"].isel(nmonth=3)
cesm1_hstot_nh_mar = ds_cesm1_hstot_nh["hs_monthly"].isel(nmonth=3)

cesm1_aicetot_nh_sep = ds_cesm1_aicetot_nh["aice_monthly"].isel(nmonth=9)
cesm1_hitot_nh_sep = ds_cesm1_hitot_nh["hi_monthly"].isel(nmonth=9)
cesm1_hstot_nh_sep = ds_cesm1_hstot_nh["hs_monthly"].isel(nmonth=9)

cesm1_aicetot_sh_feb = ds_cesm1_aicetot_sh["aice_monthly"].isel(nmonth=2)
cesm1_hitot_sh_feb = ds_cesm1_hitot_sh["hi_monthly"].isel(nmonth=2)
cesm1_hstot_sh_feb = ds_cesm1_hstot_sh["hs_monthly"].isel(nmonth=2)

cesm1_aicetot_sh_sep = ds_cesm1_aicetot_sh["aice_monthly"].isel(nmonth=9)
cesm1_hitot_sh_sep = ds_cesm1_hitot_sh["hi_monthly"].isel(nmonth=9)
cesm1_hstot_sh_sep = ds_cesm1_hstot_sh["hs_monthly"].isel(nmonth=9)

In [ ]:
cesm2_aicetot_nh_mar = ds_cesm2_aicetot_nh["aice_monthly"].isel(nmonth=3)
cesm2_hitot_nh_mar = ds_cesm2_hitot_nh["hi_monthly"].isel(nmonth=3)
cesm2_hstot_nh_mar = ds_cesm2_hstot_nh["hs_monthly"].isel(nmonth=3)

cesm2_aicetot_nh_sep = ds_cesm2_aicetot_nh["aice_monthly"].isel(nmonth=9)
cesm2_hitot_nh_sep = ds_cesm2_hitot_nh["hi_monthly"].isel(nmonth=9)
cesm2_hstot_nh_sep = ds_cesm2_hstot_nh["hs_monthly"].isel(nmonth=9)

cesm2_aicetot_sh_feb = ds_cesm2_aicetot_sh["aice_monthly"].isel(nmonth=2)
cesm2_hitot_sh_feb = ds_cesm2_hitot_sh["hi_monthly"].isel(nmonth=2)
cesm2_hstot_sh_feb = ds_cesm2_hstot_sh["hs_monthly"].isel(nmonth=2)

cesm2_aicetot_sh_sep = ds_cesm2_aicetot_sh["aice_monthly"].isel(nmonth=9)
cesm2_hitot_sh_sep = ds_cesm2_hitot_sh["hi_monthly"].isel(nmonth=9)
cesm2_hstot_sh_sep = ds_cesm2_hstot_sh["hs_monthly"].isel(nmonth=9)

In [ ]:
for case_name, case_data in case_info.items():
    climo_nyears = case_data["climo_nyears"]
    ds = case_data["ds"]
    ds_area_nh = (tarea * ds.aice).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT > 0
    ).sum(dim=["nj", "ni"]) * 1.0e-12
    ds_icevol_nh = (tarea * ds.hi).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT > 0
    ).sum(dim=["nj", "ni"]) * 1.0e-13
    ds_snovol_nh = (tarea * ds.hs).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT > 0
    ).sum(dim=["nj", "ni"]) * 1.0e-13
    ds_area_sh = (tarea * ds.aice).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT < 0
    ).sum(dim=["nj", "ni"]) * 1.0e-12
    ds_icevol_sh = (tarea * ds.hi).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT < 0
    ).sum(dim=["nj", "ni"]) * 1.0e-13
    ds_snovol_sh = (tarea * ds.hs).isel(time=slice(-climo_nyears * 12, None)).where(
        TLAT < 0
    ).sum(dim=["nj", "ni"]) * 1.0e-13

    ds_area_mar_nh = ds_area_nh.sel(time=(ds_area_nh.time.dt.month == 3)).load()
    ds_icevol_mar_nh = ds_icevol_nh.sel(time=(ds_icevol_nh.time.dt.month == 3)).load()
    ds_snovol_mar_nh = ds_snovol_nh.sel(time=(ds_snovol_nh.time.dt.month == 3)).load()
    ds_area_sep_nh = ds_area_nh.sel(time=(ds_area_nh.time.dt.month == 9)).load()
    ds_icevol_sep_nh = ds_icevol_nh.sel(time=(ds_icevol_nh.time.dt.month == 9)).load()
    ds_snovol_sep_nh = ds_snovol_nh.sel(time=(ds_snovol_nh.time.dt.month == 9)).load()
    ds_area_feb_sh = ds_area_sh.sel(time=(ds_area_sh.time.dt.month == 2)).load()
    ds_icevol_feb_sh = ds_icevol_sh.sel(time=(ds_icevol_sh.time.dt.month == 2)).load()
    ds_snovol_feb_sh = ds_snovol_sh.sel(time=(ds_snovol_sh.time.dt.month == 2)).load()
    ds_area_sep_sh = ds_area_sh.sel(time=(ds_area_sh.time.dt.month == 9)).load()
    ds_icevol_sep_sh = ds_icevol_sh.sel(time=(ds_icevol_sh.time.dt.month == 9)).load()
    ds_snovol_sep_sh = ds_snovol_sh.sel(time=(ds_snovol_sh.time.dt.month == 9)).load()

    case_info[case_name]["ds_area_mar_nh"] = ds_area_mar_nh
    case_info[case_name]["ds_icevol_mar_nh"] = ds_icevol_mar_nh
    case_info[case_name]["ds_snovol_mar_nh"] = ds_snovol_mar_nh
    case_info[case_name]["ds_area_sep_nh"] = ds_area_sep_nh
    case_info[case_name]["ds_icevol_sep_nh"] = ds_icevol_sep_nh
    case_info[case_name]["ds_snovol_sep_nh"] = ds_snovol_sep_nh
    case_info[case_name]["ds_area_feb_sh"] = ds_area_feb_sh
    case_info[case_name]["ds_icevol_feb_sh"] = ds_icevol_feb_sh
    case_info[case_name]["ds_snovol_feb_sh"] = ds_snovol_feb_sh
    case_info[case_name]["ds_area_sep_sh"] = ds_area_sep_sh
    case_info[case_name]["ds_icevol_sep_sh"] = ds_icevol_sep_sh
    case_info[case_name]["ds_snovol_sep_sh"] = ds_snovol_sep_sh

In [ ]:
cesm1_aicetot_nh_mar_mean = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_max = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_min = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_mar_std = (
    cesm1_aicetot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_nh_mar_mean = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_max = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_min = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_mar_std = (
    cesm2_aicetot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_nh_mar_mean = cdr_nh_mar.mean()
cdr_nh_mar_max = cdr_nh_mar.max()
cdr_nh_mar_min = cdr_nh_mar.min()
cdr_nh_mar_std = cdr_nh_mar.std()

In [ ]:
cesm1_aicetot_nh_sep_mean = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_max = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_min = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_nh_sep_std = (
    cesm1_aicetot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_nh_sep_mean = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_max = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_min = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_nh_sep_std = (
    cesm2_aicetot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_nh_sep_mean = cdr_nh_sep.mean()
cdr_nh_sep_max = cdr_nh_sep.max()
cdr_nh_sep_min = cdr_nh_sep.min()
cdr_nh_sep_std = cdr_nh_sep.std()

In [ ]:
cesm1_hitot_nh_mar_mean = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_max = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_min = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_mar_std = (
    cesm1_hitot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_nh_mar_mean = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_max = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_min = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_mar_std = (
    cesm2_hitot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hitot_nh_sep_mean = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_max = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_min = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_nh_sep_std = (
    cesm1_hitot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_nh_sep_mean = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_max = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_min = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_nh_sep_std = (
    cesm2_hitot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hstot_nh_mar_mean = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_max = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_min = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_mar_std = (
    cesm1_hstot_nh_mar[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_nh_mar_mean = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_max = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_min = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_mar_std = (
    cesm2_hstot_nh_mar[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hstot_nh_sep_mean = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_max = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_min = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_nh_sep_std = (
    cesm1_hstot_nh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_nh_sep_mean = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_max = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_min = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_nh_sep_std = (
    cesm2_hstot_nh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_aicetot_sh_feb_mean = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_max = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_min = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_feb_std = (
    cesm1_aicetot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_sh_feb_mean = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_max = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_min = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_feb_std = (
    cesm2_aicetot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_sh_feb_mean = cdr_sh_feb.mean()
cdr_sh_feb_max = cdr_sh_feb.max()
cdr_sh_feb_min = cdr_sh_feb.min()
cdr_sh_feb_std = cdr_sh_feb.std()

In [ ]:
cesm1_aicetot_sh_sep_mean = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_max = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_min = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm1_aicetot_sh_sep_std = (
    cesm1_aicetot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cesm2_aicetot_sh_sep_mean = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_max = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_min = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-12
)
cesm2_aicetot_sh_sep_std = (
    cesm2_aicetot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-12
)

cdr_sh_sep_mean = cdr_sh_sep.mean()
cdr_sh_sep_max = cdr_sh_sep.max()
cdr_sh_sep_min = cdr_sh_sep.min()
cdr_sh_sep_std = cdr_sh_sep.std()

In [ ]:
cesm1_hitot_sh_feb_mean = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_max = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_min = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_feb_std = (
    cesm1_hitot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_sh_feb_mean = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_max = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_min = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_feb_std = (
    cesm2_hitot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hitot_sh_sep_mean = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_max = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_min = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hitot_sh_sep_std = (
    cesm1_hitot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hitot_sh_sep_mean = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_max = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_min = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hitot_sh_sep_std = (
    cesm2_hitot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hstot_sh_feb_mean = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_max = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_min = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_feb_std = (
    cesm1_hstot_sh_feb[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_sh_feb_mean = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_max = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_min = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_feb_std = (
    cesm2_hstot_sh_feb[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
cesm1_hstot_sh_sep_mean = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_max = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_min = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm1_hstot_sh_sep_std = (
    cesm1_hstot_sh_sep[:, cesm1_climo_start:cesm1_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

cesm2_hstot_sh_sep_mean = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .mean(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_max = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .max(dim="nyr")
    .max(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_min = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .min(dim="nyr")
    .min(dim="n_members")
    * 1.0e-13
)
cesm2_hstot_sh_sep_std = (
    cesm2_hstot_sh_sep[:, cesm2_climo_start:cesm2_climo_end]
    .mean(dim="nyr")
    .std(dim="n_members")
    * 1.0e-13
)

In [ ]:
import pandas as pd

data2 = None
columns = []

for case_name, case_data in case_info.items():

    ds_area_mar_nh = case_data["ds_area_mar_nh"]
    ds_area_sep_nh = case_data["ds_area_sep_nh"]
    ds_icevol_mar_nh = case_data["ds_icevol_mar_nh"]
    ds_icevol_sep_nh = case_data["ds_icevol_sep_nh"]
    ds_snovol_mar_nh = case_data["ds_snovol_mar_nh"]
    ds_snovol_sep_nh = case_data["ds_snovol_sep_nh"]

    ds_area_mar_nh_mean = ds_area_mar_nh.mean(dim="time")
    ds_area_sep_nh_mean = ds_area_sep_nh.mean(dim="time")
    ds_icevol_mar_nh_mean = ds_icevol_mar_nh.mean(dim="time")
    ds_icevol_sep_nh_mean = ds_icevol_sep_nh.mean(dim="time")
    ds_snovol_mar_nh_mean = ds_snovol_mar_nh.mean(dim="time")
    ds_snovol_sep_nh_mean = ds_snovol_sep_nh.mean(dim="time")

    cesm1_area_mar_nh_diff = ds_area_mar_nh_mean - cesm1_aicetot_nh_mar_mean
    cesm1_area_sep_nh_diff = ds_area_sep_nh_mean - cesm1_aicetot_nh_sep_mean
    cdr_area_mar_nh_diff = ds_area_mar_nh_mean - cdr_nh_mar_mean
    cdr_area_sep_nh_diff = ds_area_sep_nh_mean - cdr_nh_sep_mean
    cesm1_icevol_mar_nh_diff = ds_icevol_mar_nh_mean - cesm1_hitot_nh_mar_mean
    cesm1_icevol_sep_nh_diff = ds_icevol_sep_nh_mean - cesm1_hitot_nh_sep_mean
    cesm1_snovol_mar_nh_diff = ds_snovol_mar_nh_mean - cesm1_hstot_nh_mar_mean
    cesm1_snovol_sep_nh_diff = ds_snovol_sep_nh_mean - cesm1_hstot_nh_sep_mean
    cesm2_snovol_mar_nh_diff = ds_snovol_mar_nh_mean - cesm2_hstot_nh_mar_mean
    cesm2_snovol_sep_nh_diff = ds_snovol_sep_nh_mean - cesm2_hstot_nh_sep_mean

    data = [
        [
            (cesm1_area_mar_nh_diff).values,
            (cesm1_area_mar_nh_diff / cesm1_aicetot_nh_mar_std).values,
        ],
        [
            (cesm1_area_sep_nh_diff).values,
            (cesm1_area_sep_nh_diff / cesm1_aicetot_nh_sep_std).values,
        ],
        [
            (cdr_area_mar_nh_diff).values,
            (cdr_area_mar_nh_diff / cdr_nh_mar_std).values,
        ],
        [
            (cdr_area_sep_nh_diff).values,
            (cdr_area_sep_nh_diff / cdr_nh_sep_std).values,
        ],
        [
            (cesm1_icevol_mar_nh_diff).values,
            (cesm1_icevol_mar_nh_diff / cesm1_hitot_nh_mar_std).values,
        ],
        [
            (cesm1_icevol_sep_nh_diff).values,
            (cesm1_icevol_sep_nh_diff / cesm1_hitot_nh_sep_std).values,
        ],
        [
            (cesm2_snovol_mar_nh_diff).values,
            (cesm2_snovol_mar_nh_diff / cesm2_hstot_nh_mar_std).values,
        ],
        [
            (cesm2_snovol_sep_nh_diff).values,
            (cesm2_snovol_sep_nh_diff / cesm2_hstot_nh_sep_std).values,
        ],
    ]

    case_nickname = case_info[case_name]["case_nickname"]

    if data2 is None:
        data2 = data
    else:
        for n, row in enumerate(data):
            data2[n] = data2[n] + row

    list1 = [
        case_nickname + "(diff)",
        case_nickname + "(std)",
    ]
    columns = columns + list1


print(data2)
print(columns)

df = pd.DataFrame(
    data=data2,
    columns=columns,
    index=[
        "Area vs CESM1 Mar",
        "Area vs CESM1 Sep",
        "Area vs CDR Mar",
        "Area vs CDR Sep",
        "Ice Volume vs CESM1 Mar",
        "Ice Volume vs CESM1 Sep",
        "Snow Volume vs CESM2 Mar",
        "Snow Volume vs CESM2 Sep",
    ],
)

from IPython.core.display import HTML
from IPython.display import display


def color_big_std(val):
    color = "white" if abs(val) < 3.0 else "cyan"
    return f"background-color: {color}"


styler = (
    df.style.map(color_big_std)
    .set_caption("Northern Hemisphere")
    .format("{:5.2f}".format)
)

display(HTML(styler.to_html(index=True)))

# df.to_csv("seaice.csv")

In [ ]:
data2 = None
columns = []

for case_name, case_data in case_info.items():

    ds_area_feb_sh = case_data["ds_area_feb_sh"]
    ds_area_sep_sh = case_data["ds_area_sep_sh"]
    ds_icevol_feb_sh = case_data["ds_icevol_feb_sh"]
    ds_icevol_sep_sh = case_data["ds_icevol_sep_sh"]
    ds_snovol_feb_sh = case_data["ds_snovol_feb_sh"]
    ds_snovol_sep_sh = case_data["ds_snovol_sep_sh"]

    ds_area_feb_sh_mean = ds_area_feb_sh.mean(dim="time")
    ds_area_sep_sh_mean = ds_area_sep_sh.mean(dim="time")
    ds_icevol_feb_sh_mean = ds_icevol_feb_sh.mean(dim="time")
    ds_icevol_sep_sh_mean = ds_icevol_sep_sh.mean(dim="time")
    ds_snovol_feb_sh_mean = ds_snovol_feb_sh.mean(dim="time")
    ds_snovol_sep_sh_mean = ds_snovol_sep_sh.mean(dim="time")

    cesm1_area_feb_sh_diff = ds_area_feb_sh_mean - cesm1_aicetot_sh_feb_mean
    cesm1_area_sep_sh_diff = ds_area_sep_sh_mean - cesm1_aicetot_sh_sep_mean
    cdr_area_feb_sh_diff = ds_area_feb_sh_mean - cdr_sh_feb_mean
    cdr_area_sep_sh_diff = ds_area_sep_sh_mean - cdr_sh_sep_mean
    cesm1_icevol_feb_sh_diff = ds_icevol_feb_sh_mean - cesm1_hitot_sh_feb_mean
    cesm1_icevol_sep_sh_diff = ds_icevol_sep_sh_mean - cesm1_hitot_sh_sep_mean
    cesm1_snovol_feb_sh_diff = ds_snovol_feb_sh_mean - cesm1_hstot_sh_feb_mean
    cesm1_snovol_sep_sh_diff = ds_snovol_sep_sh_mean - cesm1_hstot_sh_sep_mean
    cesm2_snovol_feb_sh_diff = ds_snovol_feb_sh_mean - cesm2_hstot_sh_feb_mean
    cesm2_snovol_sep_sh_diff = ds_snovol_sep_sh_mean - cesm2_hstot_sh_sep_mean

    data = [
        [
            (cesm1_area_feb_sh_diff).values,
            (cesm1_area_feb_sh_diff / cesm1_aicetot_sh_feb_std).values,
        ],
        [
            (cesm1_area_sep_sh_diff).values,
            (cesm1_area_sep_sh_diff / cesm1_aicetot_sh_sep_std).values,
        ],
        [
            (cdr_area_feb_sh_diff).values,
            (cdr_area_feb_sh_diff / cdr_sh_feb_std).values,
        ],
        [
            (cdr_area_sep_sh_diff).values,
            (cdr_area_sep_sh_diff / cdr_sh_sep_std).values,
        ],
        [
            (cesm1_icevol_feb_sh_diff).values,
            (cesm1_icevol_feb_sh_diff / cesm1_hitot_sh_feb_std).values,
        ],
        [
            (cesm1_icevol_sep_sh_diff).values,
            (cesm1_icevol_sep_sh_diff / cesm1_hitot_sh_sep_std).values,
        ],
        [
            (cesm2_snovol_feb_sh_diff).values,
            (cesm2_snovol_feb_sh_diff / cesm2_hstot_sh_feb_std).values,
        ],
        [
            (cesm2_snovol_sep_sh_diff).values,
            (cesm2_snovol_sep_sh_diff / cesm2_hstot_sh_sep_std).values,
        ],
    ]

    case_nickname = case_info[case_name]["case_nickname"]

    if data2 is None:
        data2 = data
    else:
        for n, row in enumerate(data):
            data2[n] = data2[n] + row

    list1 = [
        case_nickname + "(diff)",
        case_nickname + "(std)",
    ]
    columns = columns + list1


print(data2)
print(columns)

df = pd.DataFrame(
    data=data2,
    columns=columns,
    index=[
        "Area vs CESM1 Feb",
        "Area vs CESM1 Sep",
        "Area vs CDR Feb",
        "Area vs CDR Sep",
        "Ice Volume vs CESM1 Feb",
        "Ice Volume vs CESM1 Sep",
        "Snow Volume vs CESM2 Feb",
        "Snow Volume vs CESM2 Sep",
    ],
)

from IPython.core.display import HTML
from IPython.display import display


def color_big_std(val):
    color = "white" if abs(val) < 3.0 else "cyan"
    return f"background-color: {color}"


styler = (
    df.style.map(color_big_std)
    .set_caption("Southern Hemisphere")
    .format("{:5.2f}".format)
)

display(HTML(styler.to_html(index=True)))

# df.to_csv("seaice.csv")

In [ ]:
if client and not cupid_run:
    client.shutdown()